In [2]:
# Importações e Inicialização da SparkSession
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("Formatos-Spark-JSONL")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.spark:spark-avro_2.12:3.5.0")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "root-minio")
    .config("spark.hadoop.fs.s3a.secret.key", "root12345678")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

print(f"SparkSession ativa. Versao: {spark.version}")

SparkSession ativa. Versao: 3.5.0


In [3]:
# Extração (Extract) do PostgreSQL via JDBC
print("Extraindo eventos de telemetria com cargas JSONB via JDBC...")

df_eventos = (spark.read.jdbc(
    url="jdbc:postgresql://postgres:5432/postgres",
    table="eventos_web",
    properties={"user": "postgres", "password": "senha123", "driver": "org.postgresql.Driver"}
))

df_eventos.printSchema()

Extraindo eventos de telemetria com cargas JSONB via JDBC...
root
 |-- id_evento: string (nullable = true)
 |-- timestamp_evento: timestamp (nullable = true)
 |-- tipo_evento: string (nullable = true)
 |-- payload: string (nullable = true)
 |-- data_criacao: timestamp (nullable = true)



In [5]:
# Carga (Load) para o MinIO em formato JSONL
bucket_name = "raw"
destino_s3 = f"s3a://{bucket_name}/spark_jobs/telemetria_jsonl/"

print(f"Gravando dados em formato JSON Lines no Object Storage: {destino_s3}")

# O metodo .json() do Spark converte as linhas em strings estruturadas JSONL nativamente
df_eventos.write.mode("overwrite").json(destino_s3)

print("Escrita do JSONL concluida com sucesso no destino.")

Gravando dados em formato JSON Lines no Object Storage: s3a://raw/spark_jobs/telemetria_jsonl/
Escrita do JSONL concluida com sucesso no destino.


In [6]:
# Auditoria e Validacao dos Dados Gravados
print("Lendo arquivos JSONL de volta do Data Lake para validacao...")

df_validacao = spark.read.json(destino_s3)

print(f"Quantidade total de registros localizados: {df_validacao.count()}")
df_validacao.show(5, truncate=False)

Lendo arquivos JSONL de volta do Data Lake para validacao...
Quantidade total de registros localizados: 50000
+------------------------+------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------+-----------+
|data_criacao            |id_evento                           |payload                                                                                                                                                                                                                       |timestamp_evento        |tipo_evento|
+------------------------+------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
# Encerramento da Sessão e Limpeza de Recursos
print("Encerrando a SparkSession e liberando os recursos da JVM...")
spark.stop()
print("Sessao fechada com seguranca.")

Encerrando a SparkSession e liberando os recursos da JVM...
Sessao fechada com seguranca.


### Análise de Arquitetura: O Formato CSV no Apache Spark

**Características Principais:** Formato baseado em texto plano, estritamente orientado a linhas.

#### Vantagens
* **Universalidade:** É o formato mais democrático do mercado. Qualquer ferramenta, desde o Microsoft Excel até sistemas legados em C# ou COBOL, consegue ler um arquivo CSV sem necessidade de bibliotecas complexas.
* **Transparência:** Por ser texto puro, pode ser aberto e inspecionado visualmente em qualquer editor de blocos de notas para depuração rápida.

#### Desvantagens
* **Overhead de Processamento:** O Spark precisa converter tipos estruturados (como inteiros, decimais e timestamps) em strings textuais na escrita, e fazer o processo inverso (inferência de schema) na leitura, o que consome CPU.
* **Ausência de Tipagem Nativa:** O CSV não armazena metadados de esquema. Dados numéricos de alta precisão perdem o rigor técnico se não forem remapeados manualmente na leitura.
* **Impacto do Paralelismo:** Por ser uma engine distribuída, o Spark criará uma pasta e salvará múltiplos arquivos parciais (como part-00000, part-00001) dependendo de como os dados estavam distribuídos no cluster. Isso pode confundir usuários de negócios que esperam um arquivo único.

#### Casos de Uso no Mercado
* Compartilhamento final de dados agregados e tratados com equipes de negócio ou parceiros externos.
* Exportação de tabelas de parametrização estáticas e de baixíssimo volume (tabelas de De-Para).